# Aurora MySQL Bronze to Silver Normalization
This notebook materializes domain-ready silver tables from Aurora MySQL CDC bronze tables for sales orders, appointments, POS invoices, work orders, and inventory snapshots.

In [ ]:
from pyspark.sql import functions as F

BRONZE = 'bronze'
SILVER = 'silver'
CHECKPOINT_ROOT = 'dbfs:/checkpoints/aurora_bronze_to_silver'

def upsert_stream(source_table: str, transform_fn, target_table: str):
    source_df = spark.readStream.table(f'{BRONZE}.{source_table}')
    transformed_df = transform_fn(source_df)
    return (
        transformed_df.writeStream
        .format('delta')
        .outputMode('append')
        .option('checkpointLocation', f'{CHECKPOINT_ROOT}/{target_table}')
        .trigger(availableNow=True)
        .toTable(f'{SILVER}.{target_table}')
    )

In [ ]:
def transform_sales_orders(df):
    return (
        df.filter(F.col('op').isin('c', 'r', 'u'))
          .select(
              'sales_order_id',
              'store_id',
              'region',
              'customer_id',
              F.col('order_status').alias('status'),
              'order_amount',
              'brand_tier',
              'event_ts',
              'ingest_ts'
          )
    )

def transform_appointments(df):
    return (
        df.filter(F.col('op').isin('c', 'r', 'u'))
          .select(
              'appointment_id',
              'store_id',
              'region',
              'customer_id',
              F.col('appointment_status').alias('status'),
              'appointment_start_ts',
              'event_ts',
              'ingest_ts'
          )
    )

In [ ]:
def transform_pos_invoices(df):
    return (
        df.filter(F.col('op').isin('c', 'r', 'u'))
          .select(
              'invoice_id',
              'store_id',
              'region',
              'sales_order_id',
              'invoice_amount',
              'refund_flag',
              'brand_tier',
              'event_ts',
              'ingest_ts'
          )
    )

def transform_work_orders(df):
    return (
        df.filter(F.col('op').isin('c', 'r', 'u'))
          .select(
              'work_order_id',
              'store_id',
              'region',
              F.col('work_order_status').alias('status'),
              'cycle_time_minutes',
              'event_ts',
              'ingest_ts'
          )
    )

def transform_inventory_snapshots(df):
    return (
        df.filter(F.col('op').isin('c', 'r', 'u'))
          .select(
              'snapshot_id',
              'store_id',
              'region',
              'sku_count',
              'in_stock_sku_count',
              'low_stock_sku_count',
              'stockout_sku_count',
              'event_ts',
              'ingest_ts'
          )
    )

In [ ]:
queries = [
    upsert_stream('sales_orders_cdc', transform_sales_orders, 'sales_orders'),
    upsert_stream('appointments_cdc', transform_appointments, 'appointments'),
    upsert_stream('pos_invoices_cdc', transform_pos_invoices, 'pos_invoices'),
    upsert_stream('work_orders_cdc', transform_work_orders, 'work_orders'),
    upsert_stream('inventory_snapshots_cdc', transform_inventory_snapshots, 'inventory_snapshots'),
]

for query in queries:
    query.awaitTermination()